# 084 — pLDDT vs family-sensitivity gap (QfO 9-species Pfam benchmark)

**Hypothesis**: FoldSeek detects Pfam family homologs via 3Di built on predicted structure.  
Where AlphaFold is uncertain (low pLDDT) the 3Di is noisy and FoldSeek degrades.  
Kmerseek reads sequence and is prediction-independent, so its FAM should NOT track pLDDT.  
If true: FoldSeek's advantage shrinks (and reverses) as pLDDT falls.

**Ground truth**: Shared Pfam domain ID between human and each of 8 QfO species.  
**Queries**: Human proteins with AF2 models (pLDDT available via EBI).  
**Tools**:
- Kmerseek: hp k=24 (proxy for thomas_dill k=26; k=26 pbotc run blocked on Docker/arm64)
- MMseqs2 seqseq: sequence search (control; should track pLDDT weakly)
- FoldSeek: **STUB** — not yet run on QfO Pfam benchmark (arm64/Linux blocker)

**Status**: Kmerseek + MMseqs2 + pLDDT runnable now; FoldSeek cells marked ⚠️ TODO.

In [1]:
import concurrent.futures
import gzip
import re
import time
import urllib.error
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from scipy import stats

pl.Config.set_tbl_rows(10)

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
REPO        = Path('..').resolve()
PAIRS_DIR   = REPO / 'results/pfam_benchmark/pairs'
MERGED_DIR  = REPO / 'results/pfam_benchmark/merged'
NF_TOOLS    = Path.home() / 'code/2024-kmerseek-analysis/nextflow-runs/pfam-benchmark-tools'
AF2_CACHE   = Path.home() / 'data/alphafold_structures/structures'
OUT_DIR     = REPO / 'data'
PREFIX      = '084_'

# Kmerseek: hp k=24 (proxy; k=26 pbotc not yet run on QfO)
KS_KSIZE = 24
KS_SCORE = 'enrichment'  # higher = more likely true positive

# Species in the QfO pfam benchmark
SPECIES = ['mouse', 'zebrafish', 'chicken', 'fly', 'worm', 'yeast', 'arabidopsis', 'ecoli', 'ciona']

# MMseqs2 seqseq result paths (one per species, latest from nextflow work dir)
MM_PATHS = {
    'arabidopsis': NF_TOOLS / 'work/6e/5dedf6969d9356cb3594ebcd6aae6a/human_vs_arabidopsis.mmseqs2_seqseq.tsv.gz',
    'chicken':     NF_TOOLS / 'work/8c/35833b475fe22f2188cd42977e67b8/human_vs_chicken.mmseqs2_seqseq.tsv.gz',
    'ciona':       NF_TOOLS / 'work/03/1ad4583a5da2a54a5b3e9427905a40/human_vs_ciona.mmseqs2_seqseq.tsv.gz',
    'ecoli':       NF_TOOLS / 'work/15/2b6b1b79206834449adaf6813a33d5/human_vs_ecoli.mmseqs2_seqseq.tsv.gz',
    'fly':         NF_TOOLS / 'work/91/877393137c9eb8acdb029ed559b7c2/human_vs_fly.mmseqs2_seqseq.tsv.gz',
    'mouse':       NF_TOOLS / 'work/31/b812ab5d5dc272c3bf9da063aed96e/human_vs_mouse.mmseqs2_seqseq.tsv.gz',
    'worm':        NF_TOOLS / 'work/42/f90a586fdfa3f4dcdf016f1a61857f/human_vs_worm.mmseqs2_seqseq.tsv.gz',
    'yeast':       NF_TOOLS / 'work/7d/afe6c7f895d63ffbc5d48b2e316051/human_vs_yeast.mmseqs2_seqseq.tsv.gz',
    'zebrafish':   NF_TOOLS / 'work/92/b59a45ec53f75e5027e7452cc3367a/human_vs_zebrafish.mmseqs2_seqseq.tsv.gz',
}

print('Repo:', REPO)
print('Merged parquet dir:', MERGED_DIR.exists())
print('MMseqs2 paths exist:', all(p.exists() for p in MM_PATHS.values()))

Repo: /Users/olga/code/2024-kmerseek-analysis
Merged parquet dir: True
MMseqs2 paths exist: False


## 1. Per-query FAM computation

**Definition**: For each human protein, rank all species targets by score (descending).  
FAM = fraction of true positives retrieved before the first false positive.
A protein with no true positives is excluded from the analysis.

We compute FAM per (human_protein, species) pair and then average across species
where TPs exist. This gives one FAM value per human protein per tool.

In [2]:
def compute_per_query_fam(
    merged_df: pl.DataFrame,
    score_col: str,
    ascending: bool = False,
) -> pl.DataFrame:
    """
    Per-query sensitivity-to-first-FP (FAM) from a pair-level DataFrame.

    Parameters
    ----------
    merged_df : has columns [human_accession, species_accession, label, <score_col>]
    score_col : score column; higher = better match (unless ascending=True)
    ascending : set True if lower score = better (e.g. e-value)

    Returns
    -------
    DataFrame with columns: human_accession, fam_score, n_tp, n_pairs
    Only rows with at least 1 TP are returned.
    """
    rows = []
    for (acc,), group in merged_df.group_by(['human_accession']):
        n_tp = int(group['label'].sum())
        if n_tp == 0:
            continue
        ranked = group.sort(score_col, descending=not ascending)
        labels = ranked['label'].to_numpy().astype(bool)
        # Find index of first FP (first False in ranked list)
        fp_idx = next((i for i, v in enumerate(labels) if not v), None)
        if fp_idx is None:
            fam = 1.0
        else:
            fam = labels[:fp_idx].sum() / n_tp
        rows.append({
            'human_accession': acc,
            'fam_score': fam,
            'n_tp': n_tp,
            'n_pairs': len(group),
        })
    return pl.DataFrame(rows, schema={
        'human_accession': pl.Utf8,
        'fam_score': pl.Float64,
        'n_tp': pl.Int32,
        'n_pairs': pl.Int32,
    })


def per_query_fam_across_species(
    tool_dfs: dict[str, pl.DataFrame],
    score_col: str,
    ascending: bool = False,
    tool_label: str = 'tool',
) -> pl.DataFrame:
    """
    Compute per-query FAM averaged across all species for which the human protein
    has at least 1 TP. Returns one row per human protein.

    tool_dfs: dict mapping species -> pair-level DataFrame with label + score_col
    """
    per_sp = []
    for sp, df in tool_dfs.items():
        fam = compute_per_query_fam(df, score_col, ascending)
        fam = fam.with_columns(pl.lit(sp).alias('species'))
        per_sp.append(fam)

    all_sp = pl.concat(per_sp)
    # Average FAM per human protein across species where TPs existed
    agg = (
        all_sp
        .group_by('human_accession')
        .agg([
            pl.col('fam_score').mean().alias('fam_mean'),
            pl.col('fam_score').median().alias('fam_median'),
            pl.col('n_tp').sum().alias('total_tp'),
            pl.len().alias('n_species_with_tp'),
        ])
        .with_columns(pl.lit(tool_label).alias('tool'))
    )
    return agg


print('FAM functions defined.')

FAM functions defined.


### 1a. Kmerseek hp k=24 per-query FAM

In [3]:
KS_CACHE = OUT_DIR / f'{PREFIX}ks_k{KS_KSIZE}_per_query_fam.parquet'

if KS_CACHE.exists():
    ks_fam = pl.read_parquet(KS_CACHE)
    print(f'Loaded cached Kmerseek FAM: {len(ks_fam):,} human proteins')
else:
    ks_dfs = {}
    for sp in SPECIES:
        p = MERGED_DIR / f'human_vs_{sp}.hp.k{KS_KSIZE}.merged.parquet'
        if not p.exists():
            print(f'  MISSING: {p.name}')
            continue
        df = pl.read_parquet(p).select(
            ['human_accession', 'species_accession', 'label', KS_SCORE]
        )
        ks_dfs[sp] = df
        print(f'  loaded {sp}: {len(df):,} pairs, {df["label"].sum()} TPs')

    ks_fam = per_query_fam_across_species(ks_dfs, KS_SCORE, tool_label='kmerseek_hp_k24')
    ks_fam.write_parquet(KS_CACHE)
    print(f'\nKmerseek FAM: {len(ks_fam):,} unique human proteins')

print(ks_fam.describe())

  loaded mouse: 72,649 pairs, 32208 TPs
  loaded zebrafish: 69,744 pairs, 28731 TPs
  loaded chicken: 68,611 pairs, 28301 TPs
  loaded fly: 76,775 pairs, 32681 TPs
  loaded worm: 63,897 pairs, 23183 TPs
  loaded yeast: 63,466 pairs, 22543 TPs
  loaded arabidopsis: 61,474 pairs, 21391 TPs
  loaded ecoli: 49,892 pairs, 9876 TPs
  loaded ciona: 64,845 pairs, 23915 TPs

Kmerseek FAM: 12,264 unique human proteins
shape: (9, 7)
┌────────────┬────────────────┬──────────┬────────────┬───────────┬────────────────┬───────────────┐
│ statistic  ┆ human_accessio ┆ fam_mean ┆ fam_median ┆ total_tp  ┆ n_species_with ┆ tool          │
│ ---        ┆ n              ┆ ---      ┆ ---        ┆ ---       ┆ _tp            ┆ ---           │
│ str        ┆ ---            ┆ f64      ┆ f64        ┆ f64       ┆ ---            ┆ str           │
│            ┆ str            ┆          ┆            ┆           ┆ f64            ┆               │
╞════════════╪════════════════╪══════════╪════════════╪═══════════╪══

### 1b. MMseqs2 seqseq per-query FAM

In [4]:
MM_CACHE = OUT_DIR / f'{PREFIX}mmseqs2_seqseq_per_query_fam.parquet'

def extract_accession(s: str) -> str:
    m = re.search(r'(?:sp|tr)\|([A-Z0-9]+)\|', s)
    return m.group(1) if m else s.split('|')[0]


def load_mmseqs2_for_species(sp: str, gt_path: Path) -> pl.DataFrame | None:
    tsv_path = MM_PATHS.get(sp)
    if tsv_path is None or not tsv_path.exists():
        return None
    gt = pl.read_parquet(gt_path).select(['human_accession', 'species_accession', 'label'])

    rows = []
    with gzip.open(tsv_path, 'rt') as fh:
        for line in fh:
            parts = line.rstrip('\n').split('\t')
            if len(parts) < 3:
                continue
            q_acc = extract_accession(parts[0])
            t_acc = extract_accession(parts[1])
            try:
                score = float(parts[2])
            except ValueError:
                continue
            rows.append((q_acc, t_acc, score))

    hits = pl.DataFrame(rows, schema={
        'human_accession': pl.Utf8,
        'species_accession': pl.Utf8,
        'mm_score': pl.Float64,
    }).unique(subset=['human_accession', 'species_accession'], keep='last')

    # Join to full ground truth (pairs with score 0 = no hit)
    merged = gt.join(hits, on=['human_accession', 'species_accession'], how='left')
    merged = merged.with_columns(pl.col('mm_score').fill_null(0.0))
    return merged


if MM_CACHE.exists():
    mm_fam = pl.read_parquet(MM_CACHE)
    print(f'Loaded cached MMseqs2 FAM: {len(mm_fam):,} human proteins')
else:
    mm_dfs = {}
    for sp in SPECIES:
        gt_path = PAIRS_DIR / f'human_vs_{sp}_ground_truth.parquet'
        df = load_mmseqs2_for_species(sp, gt_path)
        if df is None:
            print(f'  MISSING: {sp}')
            continue
        mm_dfs[sp] = df
        print(f'  loaded {sp}: {len(df):,} pairs')

    mm_fam = per_query_fam_across_species(mm_dfs, 'mm_score', tool_label='mmseqs2_seqseq')
    mm_fam.write_parquet(MM_CACHE)
    print(f'\nMMseqs2 FAM: {len(mm_fam):,} unique human proteins')

print(mm_fam.describe())

  MISSING: mouse
  MISSING: zebrafish
  MISSING: chicken
  MISSING: fly
  MISSING: worm
  MISSING: yeast
  MISSING: arabidopsis
  MISSING: ecoli
  MISSING: ciona


ValueError: cannot concat empty list

### 1c. FoldSeek per-query FAM ⚠️ STUB — needs arm64/Linux FoldSeek rerun on QfO

FoldSeek is not yet in the `pfam-benchmark-tools` pipeline. Once run, its TSV will follow the same
3-column (query, target, bitscore) format as MMseqs2. The stub below shows how to load it.

In [ ]:
# ⚠️  STUB — FoldSeek results not yet available for QfO Pfam benchmark
# Once `pfam-benchmark-tools/main.nf` includes a `foldseekSearch` process, use:
#
#   FS_PATHS = {
#       sp: NF_TOOLS / f'work/<hash>/human_vs_{sp}.foldseek_3di.tsv.gz'
#       for sp in SPECIES
#   }
#
# Then load with load_mmseqs2_for_species() — same 3-column TSV format.
#
#   fs_fam = per_query_fam_across_species(fs_dfs, 'fs_score', tool_label='foldseek')
#
# For now, fs_fam is None — cells that require it are guarded with:
#   if fs_fam is not None: ...

fs_fam = None
print('FoldSeek FAM: NOT AVAILABLE (arm64/Linux rerun needed)')

## 2. AlphaFold2 pLDDT for human query proteins

pLDDT is stored as the B-factor in AF2 CIF files (`_atom_site.B_iso_or_equiv`).  
We use CA atoms only (one per residue) to compute mean pLDDT and fraction < 50 / < 70.

In [ ]:
PLDDT_CACHE = OUT_DIR / f'{PREFIX}human_plddt.parquet'
AF2_HUMAN_DIR = AF2_CACHE / 'human'
AF2_HUMAN_DIR.mkdir(parents=True, exist_ok=True)

AF_CIF_URL = 'https://alphafold.ebi.ac.uk/files/AF-{acc}-F1-model_v4.cif'
MISSING_PATH = AF2_HUMAN_DIR / 'missing_structures.txt'


def download_cif(acc: str, outdir: Path, skip_accs: set) -> tuple[str, str]:
    """Download AF2 CIF for acc. Returns (acc, 'ok'|'missing'|'skip')."""
    if acc in skip_accs:
        return acc, 'skip'
    dest = outdir / f'AF-{acc}-F1-model_v4.cif'
    if dest.exists() and dest.stat().st_size > 0:
        return acc, 'cached'
    url = AF_CIF_URL.format(acc=acc)
    for attempt in range(3):
        try:
            with urllib.request.urlopen(url, timeout=30) as resp:
                dest.write_bytes(resp.read())
            return acc, 'ok'
        except urllib.error.HTTPError as e:
            if e.code == 404:
                return acc, 'missing'
            time.sleep(2 ** attempt)
        except Exception:
            time.sleep(2 ** attempt)
    return acc, 'error'


def parse_plddt_from_cif(cif_path: Path) -> np.ndarray | None:
    """Extract per-residue pLDDT (B-factor of CA atoms) from AF2 CIF file."""
    vals = []
    in_atom_loop = False
    col_names = []
    b_col = None
    atom_col = None

    with open(cif_path) as fh:
        for line in fh:
            line = line.rstrip()
            if line == 'loop_':
                in_atom_loop = False
                col_names = []
                b_col = atom_col = None
                continue
            if line.startswith('_atom_site.'):
                in_atom_loop = True
                col_names.append(line.strip())
                if line.strip() == '_atom_site.B_iso_or_equiv':
                    b_col = len(col_names) - 1
                if line.strip() == '_atom_site.label_atom_id':
                    atom_col = len(col_names) - 1
                continue
            if in_atom_loop and b_col is not None and line.startswith(('ATOM', 'HETATM')):
                parts = line.split()
                if len(parts) <= max(b_col, atom_col or 0):
                    continue
                if atom_col is not None and parts[atom_col] != 'CA':
                    continue
                try:
                    vals.append(float(parts[b_col]))
                except ValueError:
                    pass
    return np.array(vals) if vals else None


def compute_plddt(cif_path: Path) -> dict | None:
    vals = parse_plddt_from_cif(cif_path)
    if vals is None or len(vals) == 0:
        return None
    return {
        'mean_plddt': float(vals.mean()),
        'frac_lt50': float((vals < 50).mean()),
        'frac_lt70': float((vals < 70).mean()),
        'n_residues': len(vals),
    }


if PLDDT_CACHE.exists():
    plddt_df = pl.read_parquet(PLDDT_CACHE)
    print(f'Loaded cached pLDDT: {len(plddt_df):,} proteins')
else:
    # Collect all unique human accessions across all species comparisons
    all_accs = set()
    for sp in SPECIES:
        gt = pl.read_parquet(PAIRS_DIR / f'human_vs_{sp}_ground_truth.parquet')
        all_accs.update(gt['human_accession'].to_list())
    print(f'Total unique human accessions: {len(all_accs):,}')

    # Load known-missing list (from previous runs)
    skip_accs = set()
    if MISSING_PATH.exists():
        skip_accs = set(MISSING_PATH.read_text().splitlines())
    print(f'  Skip list (known 404s): {len(skip_accs):,}')

    # Download missing CIFs in parallel
    to_download = [
        acc for acc in all_accs
        if acc not in skip_accs
        and not (AF2_HUMAN_DIR / f'AF-{acc}-F1-model_v4.cif').exists()
    ]
    print(f'  To download: {len(to_download):,}')

    new_missing = []
    if to_download:
        with concurrent.futures.ThreadPoolExecutor(max_workers=8) as pool:
            futs = {pool.submit(download_cif, acc, AF2_HUMAN_DIR, skip_accs): acc
                    for acc in to_download}
            for i, fut in enumerate(concurrent.futures.as_completed(futs)):
                acc, status = fut.result()
                if status == 'missing':
                    new_missing.append(acc)
                if (i + 1) % 500 == 0:
                    print(f'  ... {i+1}/{len(to_download)}')

        if new_missing:
            with open(MISSING_PATH, 'a') as f:
                f.write('\n'.join(new_missing) + '\n')
            print(f'  New missing: {len(new_missing):,} added to skip list')

    # Extract pLDDT from downloaded CIFs
    rows = []
    for acc in all_accs:
        cif_path = AF2_HUMAN_DIR / f'AF-{acc}-F1-model_v4.cif'
        if not cif_path.exists():
            continue
        result = compute_plddt(cif_path)
        if result is None:
            continue
        result['human_accession'] = acc
        rows.append(result)

    plddt_df = pl.DataFrame(rows).select(
        ['human_accession', 'mean_plddt', 'frac_lt50', 'frac_lt70', 'n_residues']
    )
    plddt_df.write_parquet(PLDDT_CACHE)
    print(f'Computed pLDDT for {len(plddt_df):,} proteins')

print('pLDDT summary:')
print(plddt_df.select(['mean_plddt', 'frac_lt50', 'frac_lt70', 'n_residues']).describe())

## 3. Merge FAM + pLDDT into analysis dataset

In [ ]:
def prep_fam(fam_df: pl.DataFrame, tool: str) -> pl.DataFrame:
    return (
        fam_df
        .select(['human_accession', 'fam_mean', 'n_species_with_tp'])
        .rename({'fam_mean': f'fam_{tool}'})
    )


# Build the merged analysis dataframe
base = plddt_df.clone()

# Join Kmerseek
base = base.join(prep_fam(ks_fam, 'km'), on='human_accession', how='left')

# Join MMseqs2
base = base.join(
    mm_fam.select(['human_accession', 'fam_mean']).rename({'fam_mean': 'fam_mm'}),
    on='human_accession', how='left'
)

# Join FoldSeek (when available)
if fs_fam is not None:
    base = base.join(
        fs_fam.select(['human_accession', 'fam_mean']).rename({'fam_mean': 'fam_fs'}),
        on='human_accession', how='left'
    )
else:
    base = base.with_columns(pl.lit(None).cast(pl.Float64).alias('fam_fs'))

# --- Matched denominator: require pLDDT + at least one tool FAM present ---
analysis = base.filter(
    pl.col('mean_plddt').is_not_null()
    & pl.col('fam_km').is_not_null()
    & pl.col('fam_mm').is_not_null()
)

# Difference columns
analysis = analysis.with_columns([
    (pl.col('fam_fs') - pl.col('fam_km')).alias('diff_fs_km'),
    (pl.col('fam_mm') - pl.col('fam_km')).alias('diff_mm_km'),
])

n_total = len(analysis)
n_low   = int((analysis['mean_plddt'] < 70).sum())
n_vlow  = int((analysis['mean_plddt'] < 50).sum())

print(f'Analysis set: {n_total:,} human proteins with pLDDT + FAM')
print(f'  mean_pLDDT < 70 : {n_low:,}  ({100*n_low/n_total:.1f}%)')
print(f'  mean_pLDDT < 50 : {n_vlow:,}  ({100*n_vlow/n_total:.1f}%)')

if n_low < 30:
    print('\n⚠️  WARNING: < 30 queries in low-pLDDT bin. Results are SUGGESTIVE only.')

if fs_fam is None:
    print('\n⚠️  FoldSeek FAM not available — FoldSeek panels will be skipped.')

analysis.head(3)

## 4. Coverage check (honesty guard)

Confirm Kmerseek coverage does NOT collapse at low pLDDT. If it does, stable FAM
at low pLDDT is an artifact of returning nothing.

In [ ]:
# Coverage = fraction of human proteins with at least 1 hit in ANY species
# We define "covered" as having fam_mean > 0 (not null, not 0 from no hits)
#
# For per-species coverage: load each species merged parquet and compute
# fraction of human proteins with enrichment > 0

bins = [0, 50, 60, 70, 80, 90, 100]
bin_labels = ['<50', '50-60', '60-70', '70-80', '80-90', '90-100']
ctr = [25, 55, 65, 75, 85, 95]


def plddt_bin(plddt: float) -> int:
    for i in range(len(bins) - 1):
        if plddt < bins[i + 1]:
            return i
    return len(bins) - 2


# Build coverage per pLDDT bin for kmerseek (fam_km is not null = had TPs and was analyzed)
# This isn't quite the same as "returned >= 1 hit" but is a proxy.
# A better metric: fraction of proteins in each pLDDT bin that have fam_km not null.

# Add bin column to the full base (before filtering to analysis set)
base_with_bin = base.with_columns([
    pl.col('mean_plddt').fill_null(-1).alias('_p')
]).with_columns([
    pl.when(pl.col('_p') < 50).then(0)
    .when(pl.col('_p') < 60).then(1)
    .when(pl.col('_p') < 70).then(2)
    .when(pl.col('_p') < 80).then(3)
    .when(pl.col('_p') < 90).then(4)
    .otherwise(5)
    .alias('plddt_bin')
])

# Proteins with pLDDT available
has_plddt = base_with_bin.filter(pl.col('mean_plddt').is_not_null())

cov_rows = []
for b in range(len(bin_labels)):
    sub = has_plddt.filter(pl.col('plddt_bin') == b)
    n_total_b = len(sub)
    if n_total_b == 0:
        continue
    n_km = int(sub['fam_km'].is_not_null().sum())
    n_mm = int(sub['fam_mm'].is_not_null().sum())
    n_fs = int(sub['fam_fs'].is_not_null().sum()) if 'fam_fs' in sub.columns else 0
    cov_rows.append({
        'bin': bin_labels[b], 'center': ctr[b], 'n_proteins': n_total_b,
        'cov_km': n_km / n_total_b,
        'cov_mm': n_mm / n_total_b,
        'cov_fs': n_fs / n_total_b if fs_fam is not None else float('nan'),
    })

cov_df = pl.DataFrame(cov_rows)
print('Coverage by pLDDT bin:')
print(cov_df)

## 5. Primary regression: (FoldSeek_FAM − Kmerseek_FAM) ~ pLDDT

⚠️ FoldSeek FAM not yet available — will run once arm64/Linux rerun completes.  
Showing MMseqs2 − Kmerseek as a partial check in the interim.

In [ ]:
def ols_report(x: np.ndarray, y: np.ndarray, label: str) -> dict:
    """OLS slope + CI + Spearman for x → y."""
    sl, ic, r, p, se = stats.linregress(x, y)
    # 95% CI on slope: t-distribution
    t_crit = stats.t.ppf(0.975, df=len(x) - 2)
    sp_r, sp_p = stats.spearmanr(x, y)
    crossover = -ic / sl if sl != 0 else float('nan')
    return {
        'model': label,
        'n': len(x),
        'slope': round(sl, 6),
        'slope_ci_lo': round(sl - t_crit * se, 6),
        'slope_ci_hi': round(sl + t_crit * se, 6),
        'intercept': round(ic, 4),
        'p_ols': float(p),
        'spearman_r': round(sp_r, 4),
        'p_spearman': float(sp_p),
        'crossover_plddt': round(crossover, 1) if 30 < crossover < 110 else None,
    }


stat_rows = []
plddt = analysis['mean_plddt'].to_numpy()

# --- Primary: FoldSeek − Kmerseek (if available) ---
if fs_fam is not None:
    mask = analysis['diff_fs_km'].is_not_null().to_numpy()
    stat_rows.append(ols_report(plddt[mask], analysis['diff_fs_km'].to_numpy()[mask],
                                'FoldSeek_FAM - Kmerseek_FAM ~ pLDDT'))
else:
    print('⚠️  FoldSeek − Kmerseek regression: SKIPPED (FoldSeek not available)')

# --- Control 1: MMseqs2 − Kmerseek ---
mask_mm = analysis['diff_mm_km'].is_not_null().to_numpy()
stat_rows.append(ols_report(plddt[mask_mm], analysis['diff_mm_km'].to_numpy()[mask_mm],
                             'MMseqs2_FAM - Kmerseek_FAM ~ pLDDT'))

# --- Per-tool slopes ---
for col, lab in [('fam_km', 'Kmerseek_FAM ~ pLDDT'), ('fam_mm', 'MMseqs2_FAM ~ pLDDT')]:
    mask_t = analysis[col].is_not_null().to_numpy()
    stat_rows.append(ols_report(plddt[mask_t], analysis[col].to_numpy()[mask_t], lab))

if fs_fam is not None:
    mask_fs = analysis['fam_fs'].is_not_null().to_numpy()
    stat_rows.append(ols_report(plddt[mask_fs], analysis['fam_fs'].to_numpy()[mask_fs],
                                'FoldSeek_FAM ~ pLDDT'))

stats_df = pl.DataFrame(stat_rows)
print('Regression results:')
print(stats_df)

## 6. Binned secondary: FAM difference ± 95% CI by pLDDT bin

In [ ]:
def bin_stats(plddt_arr: np.ndarray, y_arr: np.ndarray, bins: list) -> pl.DataFrame:
    """Mean ± 95% CI of y_arr within each pLDDT bin."""
    rows = []
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (plddt_arr >= lo) & (plddt_arr < hi)
        vals = y_arr[mask & ~np.isnan(y_arr)]
        n = len(vals)
        if n == 0:
            continue
        mean = vals.mean()
        sem = vals.std(ddof=1) / np.sqrt(n) if n > 1 else 0
        t_crit = stats.t.ppf(0.975, df=max(n - 1, 1))
        rows.append({'bin_lo': lo, 'bin_hi': hi, 'n': n, 'mean': mean,
                     'ci_lo': mean - t_crit * sem, 'ci_hi': mean + t_crit * sem})
    return pl.DataFrame(rows)


plddt_vals = analysis['mean_plddt'].to_numpy()
diff_mm = analysis['diff_mm_km'].to_numpy().astype(float)

bins5 = [0, 50, 60, 70, 80, 90, 101]
bin_mm = bin_stats(plddt_vals, diff_mm, bins5)
print('Binned MMseqs2 − Kmerseek FAM:')
print(bin_mm)

if fs_fam is not None:
    diff_fs = analysis['diff_fs_km'].to_numpy().astype(float)
    bin_fs = bin_stats(plddt_vals, diff_fs, bins5)
    print('\nBinned FoldSeek − Kmerseek FAM:')
    print(bin_fs)

## 7. Figure: 3-panel Nature style

In [ ]:
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 7,
    'axes.linewidth': 0.6,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'xtick.major.width': 0.6,
    'ytick.major.width': 0.6,
    'xtick.major.size': 2.5,
    'ytick.major.size': 2.5,
    'legend.frameon': False,
})

KM  = '#1b6e4e'   # Kmerseek — dark green
FS  = '#9467bd'   # FoldSeek — purple
MM  = '#7f7f7f'   # MMseqs2 — gray
GRID = '#e6e6e6'

fig = plt.figure(figsize=(7.2, 4.6))
gs = fig.add_gridspec(
    2, 2, width_ratios=[1.35, 1.0], height_ratios=[1, 1],
    hspace=0.55, wspace=0.38,
    left=0.085, right=0.985, top=0.88, bottom=0.11,
)

# ── Panel A: per-tool FAM vs pLDDT ──────────────────────────────────────────
axA = fig.add_subplot(gs[:, 0])
xs  = np.linspace(plddt_vals.min(), plddt_vals.max(), 120)

tool_data = [
    ('fam_km', KM,  'Kmerseek hp k=24 (this work)'),
    ('fam_mm', MM,  'MMseqs2 seqseq (sequence)'),
]
if fs_fam is not None:
    tool_data.append(('fam_fs', FS, 'FoldSeek (3Di on AF2 model)'))

for col, color, lab in tool_data:
    mask = analysis[col].is_not_null().to_numpy()
    y = analysis[col].to_numpy()[mask]
    x = plddt_vals[mask]
    axA.scatter(x, y, s=4, c=color, alpha=0.12, edgecolors='none', zorder=2)
    sl, ic, _, p, se = stats.linregress(x, y)
    t_c = stats.t.ppf(0.975, df=len(x) - 2)
    axA.plot(xs, ic + sl * xs, c=color, lw=1.8, zorder=4,
             label=f'{lab}\n  slope={sl:+.4f}/pLDDT, p={p:.2e}')

axA.set_xlabel('AlphaFold confidence  (mean pLDDT)')
axA.set_ylabel('family sensitivity  (FAM)')
axA.set_ylim(-0.05, 1.05)
axA.grid(True, color=GRID, lw=0.5, zorder=0)
axA.legend(loc='lower right', fontsize=5.4, handlelength=1.4,
           borderaxespad=0.2, labelspacing=0.9)
axA.set_title('a   Per-tool FAM vs AlphaFold confidence',
              loc='left', fontsize=8, fontweight='bold')

# ── Panel B: headline difference (FoldSeek − Kmerseek) or MMseqs2 − Kmerseek ─
axB = fig.add_subplot(gs[0, 1])
axB.axhline(0, color='k', lw=0.7, ls=(0, (4, 3)), zorder=3)

# Use FoldSeek if available, else MMseqs2 as illustrative
diff_col  = 'diff_fs_km' if fs_fam is not None else 'diff_mm_km'
diff_color = FS if fs_fam is not None else MM
diff_label = 'FoldSeek' if fs_fam is not None else 'MMseqs2'

mask_d = analysis[diff_col].is_not_null().to_numpy()
d_vals = analysis[diff_col].to_numpy()[mask_d]
x_d    = plddt_vals[mask_d]

axB.scatter(x_d, d_vals, s=4, c='#444444', alpha=0.15, edgecolors='none', zorder=2)
sl_d, ic_d, _, p_d, se_d = stats.linregress(x_d, d_vals)
t_cd = stats.t.ppf(0.975, df=len(x_d) - 2)
yhat = ic_d + sl_d * xs
n_, xbar, ssx = len(x_d), x_d.mean(), np.sum((x_d - x_d.mean()) ** 2)
resid = d_vals - (ic_d + sl_d * x_d)
s_err = np.sqrt(np.sum(resid ** 2) / (n_ - 2))
ci_band = t_cd * s_err * np.sqrt(1 / n_ + (xs - xbar) ** 2 / ssx)
axB.fill_between(xs, yhat - ci_band, yhat + ci_band,
                 color=diff_color, alpha=0.15, zorder=1)
axB.plot(xs, yhat, c=diff_color, lw=1.8, zorder=4)

# Crossover
if sl_d != 0:
    cx = -ic_d / sl_d
    if 30 < cx < 100:
        axB.axvline(cx, color='#cc3333', lw=0.9, ls=':', zorder=3)
        axB.annotate(f'crossover\n≈ {cx:.0f}',
                     xy=(cx, axB.get_ylim()[0] if axB.get_ylim()[0] > -1 else -0.3),
                     xytext=(cx - 3, -0.28), fontsize=5.6, color='#cc3333', ha='right')

axB.text(0.04, 0.95, f'{diff_label} better', transform=axB.transAxes,
         fontsize=5.6, color=diff_color, va='top')
axB.text(0.04, 0.07, 'Kmerseek better', transform=axB.transAxes,
         fontsize=5.6, color=KM, va='bottom')
axB.text(0.96, 0.06, f'slope={sl_d:+.4f}\np={p_d:.2e}',
         transform=axB.transAxes, fontsize=5.8, ha='right', va='bottom')
axB.set_ylabel(f'{diff_label} FAM − Kmerseek FAM')
axB.set_xlabel('mean pLDDT')
axB.grid(True, color=GRID, lw=0.5, zorder=0)
axB.set_title("b   FAM gap vs AlphaFold confidence",
              loc='left', fontsize=8, fontweight='bold')

# ── Panel C: coverage vs pLDDT ──────────────────────────────────────────────
axC = fig.add_subplot(gs[1, 1])
cov_pd = cov_df.to_pandas()
axC.plot(cov_pd['center'], cov_pd['cov_km'], '-o', c=KM, lw=1.6, ms=4,
         label='Kmerseek')
axC.plot(cov_pd['center'], cov_pd['cov_mm'], '-s', c=MM, lw=1.6, ms=4,
         label='MMseqs2')
if fs_fam is not None:
    axC.plot(cov_pd['center'], cov_pd['cov_fs'], '-^', c=FS, lw=1.6, ms=4,
             label='FoldSeek')
# label n per bin
for _, row in cov_pd.iterrows():
    axC.text(row['center'], max(row['cov_km'], row['cov_mm']) + 0.03,
             f'n={row["n_proteins"]}', fontsize=4.5, ha='center', color='#555555')
axC.set_ylim(0, 1.15)
axC.set_ylabel('fraction of queries\nwith ≥1 species hit (FAM defined)')
axC.set_xlabel('mean pLDDT')
axC.grid(True, color=GRID, lw=0.5, zorder=0)
axC.legend(loc='lower right', fontsize=5.8, handlelength=1.4)
axC.set_title('c   Coverage does not collapse at low pLDDT?',
              loc='left', fontsize=8, fontweight='bold')

stub_note = '' if fs_fam is not None else ' (FoldSeek STUB — arm64 rerun needed)'
fig.suptitle(
    f'pLDDT vs family sensitivity — QfO 9-species Pfam benchmark{stub_note}',
    fontsize=8, y=0.97,
)

fig_path = OUT_DIR / f'{PREFIX}plddt_vs_fam_qfo.pdf'
fig.savefig(fig_path, dpi=200, bbox_inches='tight')
fig.savefig(str(fig_path).replace('.pdf', '.png'), dpi=200, bbox_inches='tight')
print(f'Saved: {fig_path}')
plt.show()

## 8. Written readout

In [ ]:
print('=' * 70)
print('ANALYSIS READOUT — QfO Pfam pLDDT vs FAM')
print('=' * 70)
print(f'\nMatched denominator: n = {n_total:,} human proteins with pLDDT + kmerseek + mmseqs2 FAM')
print(f'  Low-confidence (mean_pLDDT < 70): n = {n_low:,}')
if n_low < 30:
    print('  ⚠️  < 30 proteins at low pLDDT — results suggestive/supplementary only')

print('\nRegression results (OLS + Spearman):')
for row in stats_df.iter_rows(named=True):
    sig = '**' if row['p_ols'] < 0.01 else ('*' if row['p_ols'] < 0.05 else 'ns')
    print(f"  {row['model']}")
    print(f"    slope = {row['slope']:+.5f}  [95% CI {row['slope_ci_lo']:+.5f}, {row['slope_ci_hi']:+.5f}]")
    print(f"    p(OLS) = {row['p_ols']:.3e} {sig}  |  "
          f"Spearman r = {row['spearman_r']:+.3f}, p = {row['p_spearman']:.3e}")
    if row['crossover_plddt']:
        print(f"    crossover pLDDT ≈ {row['crossover_plddt']}")
    print()

if fs_fam is None:
    print('⚠️  FoldSeek FAM not available — primary analysis (FoldSeek − Kmerseek ~ pLDDT)')
    print('   blocked on arm64/Linux FoldSeek rerun of QfO Pfam benchmark.')
    print('   MMseqs2 − Kmerseek shown as proxy; rerun pfam-benchmark-tools with')
    print('   FoldSeek 3Di process to complete the headline analysis.')
    print()
    print('HONEST CONCLUSION (partial): Cannot yet test the structural dissociation')
    print('hypothesis (FoldSeek degrades at low pLDDT, Kmerseek does not) because')
    print('FoldSeek results for QfO are not yet available.')
    print('MMseqs2 slope provides the "sequence-only" reference arm.')